# 수정본의 실행·해석 범위

이 노트북은 2026-10-07 코드 수정본입니다. 저장된 출력과 과거 성능 숫자는 제거했습니다. 실제 Excel 전체 처리, 모델 학습·탐색과 성능 재평가는 이번 수정에서 실행하지 않았습니다. 노트북 폴더를 작업 디렉터리로 사용하고 원자료와 `ml_validation.py`, 프로젝트 의존성을 준비한 뒤 위에서 아래로 실행해야 합니다.

`수익률`은 기존 Excel 값을 신뢰하지 않고 매수가·매도가·구매주식수·투자원금으로 다시 계산합니다. 수수료·세금·배당을 포함하지 않은 계산입니다. 이전에 열람한 2021년 이후 자료는 **과거 자료 재평가**이며, 새로 봉인된 최종 테스트가 아닙니다. 전년도 재무제표라는 이유만으로 매수 당시 공개된 정보라고 보장할 수 없으므로 공시 시점 자료의 연결은 별도 과제입니다. 입력 계약과 한계는 `NOTEBOOK-REPAIR-NOTES.md`를 확인하세요.

현재 보관된 `ALL_profit_final(개).xlsx`의 13,979행 중 3행은 매도가가 없습니다(Excel 행 942, 952, 2536). 이 파일을 그대로 읽으면 `recalculate_returns`가 의도적으로 오류를 냅니다. 결측을 0으로 바꾸거나 행을 자동 삭제해 학습을 진행하지 않습니다. 원가격·보유기간 자료를 확인해 처리 근거를 정하기 전에는 전체 학습을 재개할 수 없습니다. 초기 회계 진단 비교에서 유효 13,976행의 포지션 손익 부호 141개가 양수에서 비양수로 바뀌었습니다. 이 수치는 새 가격상승 정답의 변경 개수와 다릅니다. 이는 모델 성능 검증이 아닙니다. 근거: `saved-label-audit.json`.

## 분류 목표를 가격 상승으로 분리했습니다

현재 목표는 **수정주가 기준 가격 상승 여부**입니다. `add_price_direction_targets(recalculate_returns(frame))`로 회계 열을 유지한 뒤 `가격변화율 = 100 × (매도가 / 매수가 − 1)`, `가격상승여부 = int(매도가 > 매수가)`를 추가합니다. 모델의 정답 y는 `가격상승여부`이고, `수익률_양음`은 배분된 원금·정수 주식 수에 따른 회계 손익 부호로 보존합니다.

원자료의 0주 행 25개 중 가격이 상승한 8행은 포지션 이익 0이지만 가격상승 정답은 1입니다(`verification/zero-share-audit.json`). 수정주가는 당시 체결 가능한 실제 호가와 같다고 보장되지 않습니다. 당시 가격 없이 수정주가로 정수 주식 수를 계산한 백테스트의 현실성, 원자료의 주식분할·배당 등 보정 정의는 아직 확인하지 않았습니다. 따라서 새 목표는 기업 자체의 성장·투자 수익의 증명이 아니라 저장된 수정주가의 기간 변화입니다. 기존 포지션 이익 분류의 성능 숫자를 새 목표의 성능으로 재사용하지 않습니다.

2026-10-08: 연도 키로 지표를 맞춰 평균하고 결측 지표는 선별하지 않습니다. 각 연도 회사당 500만 원으로 만든 독립 학습용 관측이며 연속 재투자 성과가 아닙니다. 매도 결과 결측은 여전히 중단합니다. Q1 가격은 두 끝 관측값 평균입니다.


In [ ]:
import value_factor_backtest as kse_functions
import pandas as pd
import numpy as np
from ml_validation import recalculate_returns, add_price_direction_targets

In [ ]:
### 수정주가 데이터프레임

In [ ]:
# 엑셀 파일 경로 (파일 경로는 실제 경로로 수정해야 합니다)
file_path = 'KSE_FIN_DATA_2023.xlsx'

# 전체 데이터셋 로드 (파일 경로와 시트 이름을 실제 데이터에 맞게 수정하세요)
df_close = pd.read_excel(file_path, sheet_name='수정주가')

# 날짜 열을 datetime 형태로 변환
df_close['Symbol Name'] = pd.to_datetime(df_close['Symbol Name'])

In [ ]:
### 1월 첫 관측과 3월 마지막 관측 수정주가의 평균(일별 평균 아님)

In [ ]:
#기업 리스트 작성
company_columns = [col for col in df_close.columns if col != 'Symbol Name']

# 연도별 평균 주가를 저장할 빈 데이터 프레임 생성
average_prices_df = pd.DataFrame()

# 각 연도에 대해 평균 주가 계산 및 데이터 프레임에 추가
for year in range(2000, 2024):  # 연도 범위는 실제 데이터에 맞게 조정하세요
    q1_averages = kse_functions.calculate_q1_endpoint_mean(df_close, year, company_columns)
    q1_averages['Year'] = year
    average_prices_df = pd.concat([average_prices_df, pd.DataFrame([q1_averages])], ignore_index=True)
    
# 'Year' 열을 데이터 프레임의 첫 번째 열로 이동
average_prices_df = average_prices_df[['Year'] + [col for col in average_prices_df.columns if col != 'Year']]

average_prices_df['Year'] = average_prices_df['Year'].astype(int)

# 결과 데이터 프레임 확인
average_prices_df

In [ ]:
### PER

In [ ]:
### EPS 추출

In [ ]:
# 엑셀 파일 읽기df_fin = pd.read_excel(file_path)
df_fin = pd.read_excel(file_path)

# Name과 EPS 컬럼을 추출하고, 회계년으로 정렬
EPS_Pure_df = df_fin[['Name', '회계년', 'EPS']].sort_values(by=['회계년'])

# 회계년을 Year로 통일하여 전처리
EPS_Pure_df.rename(columns={'회계년': 'Year'}, inplace=True)

EPS_Pure_df

In [ ]:
### PER 계산 (추후 설명변수로 들어갈꺼니까 가공 준비)

In [ ]:
df_per = kse_functions.calculate_price_ratios(
    EPS_Pure_df, 'EPS', average_prices_df, company_columns)
df_per

In [ ]:
### 1/PER 계산

In [ ]:
df_inverse_per = kse_functions.calculate_inverse_price_ratios(
    df_per, company_columns)
df_inverse_per

In [ ]:
### 기업의 수

In [ ]:
def calculate_price_ratio_rank(inverse_ratios):
    ranked_companies_by_year = pd.DataFrame()
    for year in range(2001, 2024):
        top_companies = kse_functions.top_companies_by_inverse_rank(
            inverse_ratios, year, len(company_columns))
        top_companies['Year'] = year
        top_companies['Rank'] = range(1, len(top_companies) + 1)
        ranked_companies_by_year = pd.concat(
            [ranked_companies_by_year, top_companies], ignore_index=True)
    return ranked_companies_by_year


top_per_year = calculate_price_ratio_rank(df_inverse_per)
top_per_year

In [ ]:
close_1 = df_close.copy()

# 첫 번째 열을 인덱스로 설정
close_1.set_index(close_1.columns[0], inplace=True)

# 인덱스를 날짜 형식으로 변환
close_1.index = pd.to_datetime(close_1.index)

# 초기 설정
sale_month_imsi = 3
sale_year_imsi = 2002
purchase_month_imsi = 4 
purchase_year_imsi = 2001

# 시작 연도와 종료 연도 설정
start_year = 2001
end_year = 2022

def calculate_combined_profit(ranked_companies_by_year):
    profits = []
    for year in range(start_year, end_year + 1):
        ranked_companies_for_year = ranked_companies_by_year[
            ranked_companies_by_year['Year'] == year]
        _, profit = kse_functions.process_year_data(
            year, ranked_companies_for_year,
            5000000 * len(ranked_companies_for_year),
            purchase_year_imsi, purchase_month_imsi,
            sale_year_imsi, sale_month_imsi, close_1, base_year=start_year)
        profit = add_price_direction_targets(profit)
        profit['연도'] = year
        profits.append(profit)
    return pd.concat(profits)

In [ ]:
combined_profit = calculate_combined_profit(top_per_year)
print(combined_profit)

In [ ]:
output_filename = "PER_profit_for_tree.xlsx"
combined_profit.to_excel(output_filename, index=False)

print(f"Data saved to {output_filename}")

연도는 앞선 반복문에서 붙입니다. `recalculate_returns`가 회계 손익·수익률·수익률_양음을 계산하고, `add_price_direction_targets`가 별도 가격변화율·가격상승여부를 추가합니다. Excel에서 정답을 수동 작성하지 않으며 이전 중간 파일을 읽을 때도 두 계산을 순서대로 적용합니다.

In [ ]:
df_per_profit = add_price_direction_targets(recalculate_returns(pd.read_excel("PER_profit_for_tree.xlsx")))

def merge_values(df_profit, df_inverse, column):
    df_merged = df_profit.copy()
    for index, row in df_profit.iterrows():
        year = row['연도']
        company = row['기업명']
        value = df_inverse.loc[df_inverse['Year'] == year, company].iloc[0]
        df_merged.at[index, column] = value
    return df_merged

df_merged = merge_values(df_per_profit, df_inverse_per, '1/PER')

# 결과 확인
df_merged

In [ ]:
# 원본 데이터프레임의 복사본을 생성
df_per_profit_cleaned = df_merged.copy()

# NaN 값이 있는 행을 제거
df_per_profit_cleaned.dropna(subset=['1/PER'], inplace=True)

# 1/PER 값의 순위를 내림차순으로 계산
df_per_profit_cleaned['1/PER Rank'] = df_per_profit_cleaned.groupby('연도')['1/PER'].rank(ascending=False)

# 결과 확인
df_per_profit_cleaned

In [ ]:
output_filename = "PER_profit_final.xlsx"
df_per_profit_cleaned.to_excel(output_filename, index=False)

print(f"Data saved to {output_filename}")

In [ ]:
### PBR

In [ ]:
# 엑셀 파일 읽기df_fin = pd.read_excel(file_path)
df_fin = pd.read_excel(file_path)

# Name과 BPS 컬럼을 추출하고, 회계년으로 정렬
BPS_Pure_df = df_fin[['Name', '회계년', 'BPS']].sort_values(by=['회계년'])

# 회계년을 Year로 통일하여 전처리
BPS_Pure_df.rename(columns={'회계년': 'Year'}, inplace=True)

BPS_Pure_df

In [ ]:
df_pbr = kse_functions.calculate_price_ratios(
    BPS_Pure_df, 'BPS', average_prices_df, company_columns)
df_pbr

In [ ]:
df_inverse_pbr = kse_functions.calculate_inverse_price_ratios(
    df_pbr, company_columns)
df_inverse_pbr

In [ ]:
top_pbr_year = calculate_price_ratio_rank(df_inverse_pbr)
top_pbr_year

In [ ]:
combined_profit = calculate_combined_profit(top_pbr_year)
print(combined_profit)

In [ ]:
output_filename = "PBR_profit_for_tree.xlsx"
combined_profit.to_excel(output_filename, index=False)

print(f"Data saved to {output_filename}")

In [ ]:
df_pbr_profit = add_price_direction_targets(recalculate_returns(pd.read_excel("PBR_profit_for_tree.xlsx")))

df_merged = merge_values(df_pbr_profit, df_inverse_pbr, '1/PBR')

# 원본 데이터프레임의 복사본을 생성
df_pbr_profit_cleaned = df_merged.copy()

# NaN 값이 있는 행을 제거
df_pbr_profit_cleaned.dropna(subset=['1/PBR'], inplace=True)

# 1/PBR 값의 순위를 내림차순으로 계산
df_pbr_profit_cleaned['1/PBR Rank'] = df_pbr_profit_cleaned.groupby('연도')['1/PBR'].rank(ascending=False)

output_filename = "PBR_profit_final.xlsx"
df_pbr_profit_cleaned.to_excel(output_filename, index=False)

print(f"Data saved to {output_filename}")

In [ ]:
### PSR

In [ ]:
# Name과 SPS 컬럼을 추출하고, 회계년으로 정렬
SPS_Pure_df = df_fin[['Name', '회계년', 'SPS']].sort_values(by=['회계년'])

# 회계년을 Year로 통일하여 전처리
SPS_Pure_df.rename(columns={'회계년': 'Year'}, inplace=True)

df_psr = kse_functions.calculate_price_ratios(
    SPS_Pure_df, 'SPS', average_prices_df, company_columns)

df_inverse_psr = kse_functions.calculate_inverse_price_ratios(
    df_psr, company_columns)
    
top_psr_year = calculate_price_ratio_rank(df_inverse_psr)
    
combined_profit = calculate_combined_profit(top_psr_year)
print(combined_profit)
    
output_filename = "PSR_profit_for_tree.xlsx"
combined_profit.to_excel(output_filename, index=False)

print(f"Data saved to {output_filename}")

In [ ]:
df_psr_profit = add_price_direction_targets(recalculate_returns(pd.read_excel("PSR_profit_for_tree.xlsx")))

df_merged = merge_values(df_psr_profit, df_inverse_psr, '1/PSR')

# 원본 데이터프레임의 복사본을 생성
df_psr_profit_cleaned = df_merged.copy()

# NaN 값이 있는 행을 제거
df_psr_profit_cleaned.dropna(subset=['1/PSR'], inplace=True)

# 1/PSR 값의 순위를 내림차순으로 계산
df_psr_profit_cleaned['1/PSR Rank'] = df_psr_profit_cleaned.groupby('연도')['1/PSR'].rank(ascending=False)

output_filename = "PSR_profit_final.xlsx"
df_psr_profit_cleaned.to_excel(output_filename, index=False)

print(f"Data saved to {output_filename}")

In [ ]:
### PCR

In [ ]:
# Name과 CFPS 컬럼을 추출하고, 회계년으로 정렬
CFPS_Pure_df = df_fin[['Name', '회계년', 'CFPS']].sort_values(by=['회계년'])

# 회계년을 Year로 통일하여 전처리
CFPS_Pure_df.rename(columns={'회계년': 'Year'}, inplace=True)

df_pcr = kse_functions.calculate_price_ratios(
    CFPS_Pure_df, 'CFPS', average_prices_df, company_columns)

df_inverse_pcr = kse_functions.calculate_inverse_price_ratios(
    df_pcr, company_columns)
    
top_pcr_year = calculate_price_ratio_rank(df_inverse_pcr)
    
combined_profit = calculate_combined_profit(top_pcr_year)
print(combined_profit)
    
output_filename = "PCR_profit_for_tree.xlsx"
combined_profit.to_excel(output_filename, index=False)

print(f"Data saved to {output_filename}")

In [ ]:
df_pcr_profit = add_price_direction_targets(recalculate_returns(pd.read_excel("PCR_profit_for_tree.xlsx")))

df_merged = merge_values(df_pcr_profit, df_inverse_pcr, '1/PCR')

# 원본 데이터프레임의 복사본을 생성
df_pcr_profit_cleaned = df_merged.copy()

# NaN 값이 있는 행을 제거
df_pcr_profit_cleaned.dropna(subset=['1/PCR'], inplace=True)

# 1/PCR 값의 순위를 내림차순으로 계산
df_pcr_profit_cleaned['1/PCR Rank'] = df_pcr_profit_cleaned.groupby('연도')['1/PCR'].rank(ascending=False)

output_filename = "PCR_profit_final.xlsx"
df_pcr_profit_cleaned.to_excel(output_filename, index=False)

print(f"Data saved to {output_filename}")

In [ ]:
### 동일비중 전략

In [ ]:
# 모든 지표 데이터 프레임의 리스트
data_frames = [df_inverse_per, df_inverse_pbr, df_inverse_psr, df_inverse_pcr]

df_inverse_all = kse_functions.calculate_weighted_average(data_frames)

top_all_year = calculate_price_ratio_rank(df_inverse_all)
    
combined_profit = calculate_combined_profit(top_all_year)
print(combined_profit)
    
output_filename = "ALL_profit_for_tree.xlsx"
combined_profit.to_excel(output_filename, index=False)

print(f"Data saved to {output_filename}")

In [ ]:
df_all_profit = add_price_direction_targets(recalculate_returns(pd.read_excel("ALL_profit_for_tree.xlsx")))

df_merged = merge_values(df_all_profit, df_inverse_per, '1/PER')
df_merged = merge_values(df_merged, df_inverse_pbr, '1/PBR')
df_merged = merge_values(df_merged, df_inverse_psr, '1/PSR')
df_merged = merge_values(df_merged, df_inverse_pcr, '1/PCR')
df_merged = merge_values(df_merged, df_inverse_all, 'ALL')

# 원본 데이터프레임의 복사본을 생성
df_all_profit_cleaned = df_merged.copy()

# NaN 값이 있는 행을 제거
df_all_profit_cleaned.dropna(subset=['ALL'], inplace=True)

# 값의 순위를 내림차순으로 계산
df_all_profit_cleaned['ALL Rank'] = df_all_profit_cleaned.groupby('연도')['ALL'].rank(ascending=False)

output_filename = "ALL_profit_final.xlsx"
df_all_profit_cleaned.to_excel(output_filename, index=False)

print(f"Data saved to {output_filename}")

In [ ]:
### 네가지 지표를 모두 활용하는 경우 지표별 순위를 기준으로 측정

In [ ]:
df_all_profit_cleaned['1/PER Rank'] = df_all_profit_cleaned.groupby('연도')['1/PER'].rank(ascending=False)
df_all_profit_cleaned['1/PBR Rank'] = df_all_profit_cleaned.groupby('연도')['1/PBR'].rank(ascending=False)
df_all_profit_cleaned['1/PSR Rank'] = df_all_profit_cleaned.groupby('연도')['1/PSR'].rank(ascending=False)
df_all_profit_cleaned['1/PCR Rank'] = df_all_profit_cleaned.groupby('연도')['1/PCR'].rank(ascending=False)
# 값의 순위를 내림차순으로 계산
df_all_profit_cleaned['ALL Rank'] = df_all_profit_cleaned.groupby('연도')['ALL'].rank(ascending=False)

df_all_profit_cleaned

In [ ]:
output_filename = "ALL_profit_final(개).xlsx"
df_all_profit_cleaned.to_excel(output_filename, index=False)

print(f"Data saved to {output_filename}")